In [ ]:
"""
World Cup 2026 Simulation Engine
=================================
Uses real pre-tournament data from test.csv (48 official WC 2026 teams).

Pipeline:
  1. Train XGBoost on train.csv  (2002–2022 historical data)
  2. Load test.csv               (real 2026 team stats)
  3. Build N×N win-probability matrix (single batch predict_proba call)
  4. Simulate group stage  (12 groups × 4 teams, top-2 + best-8 3rd = 32)
  5. Simulate knockout bracket (R32 → R16 → QF → SF → Final)
  6. Monte Carlo (5 000 runs) → championship probabilities

Requirements:
    pip install pandas numpy xgboost scikit-learn
"""

import pandas as pd
import numpy as np
from itertools import combinations
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — FEATURE DEFINITIONS
# ─────────────────────────────────────────────────────────────────────────────

TEAM_FEATURES = [
    "goals_scored_last_4y",
    "goals_received_last_4y",
    "wins_last_4y",
    "losses_last_4y",
    "draws_last_4y",
    "world_cup_titles_before",
    "squad_total_market_value_eur",
    "fifa_rank_pre_tournament",        # lower = better; sign is flipped in diff
    "fifa_points_pre_tournament",
    "squad_avg_age",
    "world_cup_participations_before",
    "groups_passed_before",
    "round16_before",
    "quarterfinals_before",
    "semifinals_before",
    "finals_before",
    "is_host",
]

DIFF_FEATURES = ["diff_" + f for f in TEAM_FEATURES]


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — BUILD MATCH-LEVEL DATASET FROM train.csv
# ─────────────────────────────────────────────────────────────────────────────

def performance_score(row):
    """Ordinal achievement label: used to decide who 'won' a synthetic match."""
    if row["winner"]:           return 7
    if row["finalist"]:         return 6
    if row["semi_finalist"]:    return 5
    if row["quarter_finalist"]: return 4
    return int(row["groups_passed_before"] > 0)


def build_match_dataset(train_path="train.csv"):
    """
    Load historical team data and explode into pairwise match rows.
    Each row = diff-features between two teams in the same tournament.
    Label = 1 if team_a finished further than team_b.
    Mirror rows are added to keep the dataset symmetric.
    """
    df = pd.read_csv(train_path)
    print("Loaded {} team-tournament rows, years: {}".format(
        len(df), sorted(df["version"].unique())))

    df["perf_score"] = df.apply(performance_score, axis=1)
    rows = []

    for year, group in df.groupby("version"):
        teams = group.set_index("team")
        for t_a, t_b in combinations(teams.index, 2):
            a, b = teams.loc[t_a], teams.loc[t_b]
            row = {}
            for feat in TEAM_FEATURES:
                diff = a[feat] - b[feat]
                if feat == "fifa_rank_pre_tournament":
                    diff = -diff          # flip: lower rank = better
                row["diff_" + feat] = diff
            row["label"] = int(a["perf_score"] > b["perf_score"])
            rows.append(row)

            # mirror
            mirror = {k: -v for k, v in row.items() if k != "label"}
            mirror["label"] = 1 - row["label"]
            rows.append(mirror)

    matches = pd.DataFrame(rows)
    print("Built {} match-level rows".format(len(matches)))
    return matches


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — TRAIN XGBOOST MODEL
# ─────────────────────────────────────────────────────────────────────────────

def train_model(matches):
    """Train a calibrated XGBoost classifier on the match dataset."""
    X = matches[DIFF_FEATURES].fillna(0)
    y = matches["label"]

    base = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=42,
    )

    cv_auc = cross_val_score(base, X, y, cv=5, scoring="roc_auc")
    print("XGBoost 5-fold CV AUC: {:.3f} +/- {:.3f}".format(
        cv_auc.mean(), cv_auc.std()))

    # Isotonic calibration ensures predicted probabilities are reliable
    model = CalibratedClassifierCV(base, cv=5, method="isotonic")
    model.fit(X, y)
    return model


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — LOAD REAL 2026 TEAM DATA FROM test.csv
# ─────────────────────────────────────────────────────────────────────────────

def load_test_teams(test_path="test.csv"):
    """
    Load the 48 official WC 2026 teams from test.csv.
    Returns a dict:  {team_name: {feature: value, ...}}
    The target columns (winner / finalist / semi_finalist / quarter_finalist)
    are all NaN in this file — the tournament hasn't been played yet.
    """
    df = pd.read_csv(test_path)
    print("Loaded {} teams from test.csv (year={})".format(
        len(df), df["version"].iloc[0]))

    teams = {}
    for _, row in df.iterrows():
        teams[row["team"]] = {f: row[f] for f in TEAM_FEATURES}
    return teams


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — PRE-COMPUTE WIN-PROBABILITY MATRIX
# ─────────────────────────────────────────────────────────────────────────────

def build_prob_matrix(model, teams_dict):
    """
    Single batch predict_proba call for all N×N team pairs.
    Returns (matrix[N,N], {team_name: index}).
    matrix[i,j] = P(team_i beats team_j).
    """
    all_teams = list(teams_dict.keys())
    N = len(all_teams)
    rows = []
    for t_a in all_teams:
        for t_b in all_teams:
            if t_a == t_b:
                rows.append({f: 0.0 for f in DIFF_FEATURES})
            else:
                row = {}
                for feat in TEAM_FEATURES:
                    diff = teams_dict[t_a][feat] - teams_dict[t_b][feat]
                    if feat == "fifa_rank_pre_tournament":
                        diff = -diff
                    row["diff_" + feat] = diff
                rows.append(row)

    X = pd.DataFrame(rows)[DIFF_FEATURES].fillna(0)
    flat = model.predict_proba(X)[:, 1]
    matrix = flat.reshape(N, N)
    idx = {t: i for i, t in enumerate(all_teams)}
    return matrix, idx


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — WC 2026 GROUPS  (12 groups × 4 teams)
# ─────────────────────────────────────────────────────────────────────────────
# Official / projected WC 2026 group draw.
# Team names must match exactly what appears in test.csv.

WC2026_GROUPS = {
    "A": ["Mexico", "South Africa",       "South Korea",              "Czech Republic"],
    "B": ["Canada",     "Bosnia and Herzegovina",         "Qatar",          "Switzerland"],   # placeholder CONMEBOL
    "C": ["Brazil",        "Morocco",       "Haiti",         "Scotland"],
    "D": ["United States",  "Paraguay",       "Australia",             "Turkey"],
    "E": ["Germany",       "Cura?o",   "Ivory Coast",                "Ecuador"],
    "F": ["Netherlands",       "Japan",   "Sweden",              "Tunisia"],
    "G": ["Belgium",        "Egypt",      "Iran",             "New Zealand"],
    "H": ["Spain",      "Cape Verde",       "Saudi Arabia",              "Uruguay"],
    "I": ["France",     "Senegal",       "Iraq",               "Norway"],  # duplicated fix below
    "J": ["Argentina",       "Algeria",        "Austria",             "Jordan"],
    "K": ["Portugal",   "DR Congo",         "Uzbekistan",            "Colombia"],
    "L": ["England",         "Croatia",      "Ghana",               "Panama"],
}


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 — GROUP STAGE SIMULATION
# ─────────────────────────────────────────────────────────────────────────────

def simulate_group_stage(prob_matrix, idx, groups, rng):
    """
    Round-robin within each group.
    Draw probability scales with match closeness (10–25%).
    Returns {group_letter: [(team, {pts, gd}), ...]} sorted by standing.
    """
    results = {}
    for g, members in groups.items():
        standing = {t: {"pts": 0, "gd": 0} for t in members}
        for t_a, t_b in combinations(members, 2):
            p_a = prob_matrix[idx[t_a], idx[t_b]]
            draw_p = 0.20 * (1 - abs(p_a - 0.5) * 2)
            p_a -= draw_p / 2
            roll = rng.random()
            if roll < p_a:
                standing[t_a]["pts"] += 3
                standing[t_a]["gd"]  += 1
                standing[t_b]["gd"]  -= 1
            elif roll < p_a + draw_p:
                standing[t_a]["pts"] += 1
                standing[t_b]["pts"] += 1
            else:
                standing[t_b]["pts"] += 3
                standing[t_b]["gd"]  += 1
                standing[t_a]["gd"]  -= 1

        ranked = sorted(
            members,
            key=lambda t: (standing[t]["pts"], standing[t]["gd"]),
            reverse=True,
        )
        results[g] = [(t, standing[t]) for t in ranked]
    return results


def get_qualified_teams(group_results):
    """
    Top 2 from each of 12 groups = 24.
    Best 8 third-place finishers (by pts then gd) = 8 more.
    Total = 32 for the knockout stage.
    """
    top2   = []
    thirds = []
    for g, standings in group_results.items():
        top2.append(standings[0][0])
        top2.append(standings[1][0])
        thirds.append((
            standings[2][0],
            standings[2][1]["pts"],
            standings[2][1]["gd"],
        ))

    best8 = sorted(thirds, key=lambda x: (x[1], x[2]), reverse=True)[:8]
    return top2 + [x[0] for x in best8]


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 — KNOCKOUT STAGE SIMULATION
# ─────────────────────────────────────────────────────────────────────────────

def simulate_knockout(prob_matrix, idx, teams_32, rng, verbose=False):
    """
    Single-elimination from 32 teams.
    Returns champion name.
    If verbose=True, prints each match result.
    """
    current = list(teams_32)
    stage_names = {
        32: "ROUND OF 32",
        16: "ROUND OF 16",
         8: "QUARTERFINALS",
         4: "SEMIFINALS",
         2: "FINAL",
    }

    while len(current) > 1:
        name = stage_names.get(len(current), "ROUND OF {}".format(len(current)))
        if verbose:
            print("\n── {} ──".format(name))
        rng.shuffle(current)
        next_r = []
        for i in range(0, len(current) - 1, 2):
            t_a, t_b = current[i], current[i + 1]
            p = prob_matrix[idx[t_a], idx[t_b]]
            winner = t_a if rng.random() < p else t_b
            if verbose:
                print("  {:30s} vs {:30s}  ->  {}".format(t_a, t_b, winner))
            next_r.append(winner)
        # handle odd team (bye — very rare)
        if len(current) % 2 == 1:
            next_r.append(current[-1])
        current = next_r

    return current[0]


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9 — SINGLE VIVID SIMULATION (for display)
# ─────────────────────────────────────────────────────────────────────────────

def print_single_simulation(prob_matrix, idx, groups, seed=42):
    rng = np.random.default_rng(seed)

    print("\n" + "=" * 66)
    print("   WORLD CUP 2026 — SINGLE TOURNAMENT SIMULATION  (seed={})".format(seed))
    print("=" * 66)

    # Group stage
    print("\n── GROUP STAGE ──")
    group_res = simulate_group_stage(prob_matrix, idx, groups, rng)
    for g, standings in group_res.items():
        parts = ["{} ({}pts)".format(t, s["pts"]) for t, s in standings]
        print("  Group {}: {}".format(g, "  |  ".join(parts)))

    # Qualification
    qualified = get_qualified_teams(group_res)
    # Deduplicate and pad to 32
    seen = set()
    unique_q = []
    for t in qualified:
        if t not in seen:
            seen.add(t)
            unique_q.append(t)
    if len(unique_q) < 32:
        extras = [t for t in idx if t not in seen]
        rng.shuffle(extras)
        unique_q += extras[:32 - len(unique_q)]
    qualified = unique_q[:32]

    print("\n── R32 QUALIFIERS ({} teams) ──".format(len(qualified)))
    for i in range(0, len(qualified), 4):
        print("  " + "  |  ".join(qualified[i:i+4]))

    # Knockout
    simulate_knockout(prob_matrix, idx, qualified, rng, verbose=True)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 10 — MONTE CARLO SIMULATION
# ─────────────────────────────────────────────────────────────────────────────

def monte_carlo_simulation(prob_matrix, idx, groups, teams_dict, n=5000, seed=42):
    """
    Run n full tournament simulations.
    Returns a DataFrame with P(qualify), P(QF), P(SF), P(final), P(win).
    """
    all_teams = list(teams_dict.keys())
    rng = np.random.default_rng(seed)

    counts = {t: {"qualify": 0, "qf": 0, "sf": 0, "final": 0, "win": 0}
              for t in all_teams}

    for sim in range(n):
        # ── Group stage ──
        group_res = simulate_group_stage(prob_matrix, idx, groups, rng)
        qualified = get_qualified_teams(group_res)

        # Deduplicate + pad to exactly 32
        seen = set()
        unique_q = []
        for t in qualified:
            if t not in seen:
                seen.add(t)
                unique_q.append(t)
        if len(unique_q) < 32:
            extras = [t for t in all_teams if t not in seen]
            rng.shuffle(extras)
            unique_q += extras[:32 - len(unique_q)]
        qualified = unique_q[:32]

        for t in qualified:
            counts[t]["qualify"] += 1

        # ── Knockout: track round-by-round survivors ──
        current = qualified[:]
        stage_track = ["r32", "qf", "sf", "final", "champion"]
        stage_idx = 0

        while len(current) > 1:
            rng.shuffle(current)
            next_r = []
            for i in range(0, len(current) - 1, 2):
                t_a, t_b = current[i], current[i + 1]
                p = prob_matrix[idx[t_a], idx[t_b]]
                winner = t_a if rng.random() < p else t_b
                next_r.append(winner)
            if len(current) % 2 == 1:
                next_r.append(current[-1])

            n_survivors = len(next_r)
            if n_survivors == 8:
                for t in next_r: counts[t]["qf"] += 1
            elif n_survivors == 4:
                for t in next_r: counts[t]["sf"] += 1
            elif n_survivors == 2:
                for t in next_r: counts[t]["final"] += 1
            elif n_survivors == 1:
                counts[next_r[0]]["win"] += 1

            current = next_r

    results = pd.DataFrame([
        {
            "team":           t,
            "qualify_%":      round(counts[t]["qualify"] / n * 100, 1),
            "quarterfinal_%": round(counts[t]["qf"]      / n * 100, 1),
            "semifinal_%":    round(counts[t]["sf"]      / n * 100, 1),
            "final_%":        round(counts[t]["final"]   / n * 100, 1),
            "win_%":          round(counts[t]["win"]     / n * 100, 1),
        }
        for t in all_teams
    ]).sort_values("win_%", ascending=False).reset_index(drop=True)

    return results


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 11 — HEAD-TO-HEAD PREDICTOR (utility)
# ─────────────────────────────────────────────────────────────────────────────

def predict_match(prob_matrix, idx, team_a, team_b):
    """Print win probabilities for any two teams in the tournament."""
    p_a = prob_matrix[idx[team_a], idx[team_b]]
    p_b = 1 - p_a
    print("  {:30s}  {:.1f}%  vs  {:.1f}%  {:30s}".format(
        team_a, p_a * 100, p_b * 100, team_b))


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # ── 1. Train on historical data ──────────────────────────────────────────
    print("=" * 66)
    print("STEP 1: Building match dataset from train.csv")
    print("=" * 66)
    matches = build_match_dataset("train.csv")

    print("\nSTEP 2: Training XGBoost model")
    model = train_model(matches)

    # ── 2. Load real 2026 teams ──────────────────────────────────────────────
    print("\nSTEP 3: Loading WC 2026 teams from test.csv")
    teams_2026 = load_test_teams("test.csv")

    # ── 3. Pre-compute win-probability matrix ─────────────────────────────────
    print("\nSTEP 4: Computing win-probability matrix ({n}x{n})...".format(
        n=len(teams_2026)))
    prob_matrix, idx = build_prob_matrix(model, teams_2026)

    # ── 4. Build groups using real test.csv teams ─────────────────────────────
    groups = WC2026_GROUPS
    print("\nAuto-generated groups (pot-seeded by FIFA points):")
    for g, members in groups.items():
        print("  Group {}: {}".format(g, ", ".join(members)))

    # ── 5. Example head-to-head matchups ──────────────────────────────────────
    print("\n── EXAMPLE HEAD-TO-HEAD PROBABILITIES ──")
    matchups = [
        ("France",    "Argentina"),
        ("Spain",     "Brazil"),
        ("England",   "Germany"),
        ("Morocco",   "Portugal"),
        ("Japan",     "Netherlands"),
        ("United States", "Mexico"),
    ]
    for t_a, t_b in matchups:
        predict_match(prob_matrix, idx, t_a, t_b)

    # ── 6. Single full tournament simulation ─────────────────────────────────
    print_single_simulation(prob_matrix, idx, groups, seed=42)

    # ── 7. Monte Carlo (5 000 runs) ───────────────────────────────────────────
    N_SIMS = 5000
    print("\n" + "=" * 66)
    print("STEP 5: Monte Carlo simulation (n={})".format(N_SIMS))
    print("=" * 66)
    mc = monte_carlo_simulation(
        prob_matrix, idx, groups, teams_2026, n=N_SIMS, seed=42)

    print("\n── WC 2026 CHAMPIONSHIP WIN PROBABILITIES (all 48 teams) ──")
    print(mc.to_string(index=False))

    print("\n── TOP 12 FAVOURITES ──")
    print(mc.head(12).to_string(index=False))

    # ── 8. Save results ───────────────────────────────────────────────────────
    mc.to_csv("wc2026_probabilities.csv", index=False)
    print("\nResults saved to wc2026_probabilities.csv")

STEP 1: Building match dataset from train.csv
Loaded 192 team-tournament rows, years: [np.int64(2002), np.int64(2006), np.int64(2010), np.int64(2014), np.int64(2018), np.int64(2022)]
Built 5952 match-level rows

STEP 2: Training XGBoost model
XGBoost 5-fold CV AUC: 0.757 +/- 0.036

STEP 3: Loading WC 2026 teams from test.csv
Loaded 48 teams from test.csv (year=2026)

STEP 4: Computing win-probability matrix (48x48)...

Auto-generated groups (pot-seeded by FIFA points):
  Group A: Mexico, South Africa, South Korea, Czech Republic
  Group B: Canada, Bosnia and Herzegovina, Qatar, Switzerland
  Group C: Brazil, Morocco, Haiti, Scotland
  Group D: United States, Paraguay, Australia, Turkey
  Group E: Germany, Cura?o, Ivory Coast, Ecuador
  Group F: Netherlands, Japan, Sweden, Tunisia
  Group G: Belgium, Egypt, Iran, New Zealand
  Group H: Spain, Cape Verde, Saudi Arabia, Uruguay
  Group I: France, Senegal, Iraq, Norway
  Group J: Argentina, Algeria, Austria, Jordan
  Group K: Portugal, DR 

In [ ]:
import pandas as pd

df = pd.read_csv("test.csv")

df

,version,team,continent,is_host,goals_scored_last_4y,goals_received_last_4y,wins_last_4y,losses_last_4y,draws_last_4y,world_cup_titles_before,...,world_cup_participations_before,groups_passed_before,round16_before,quarterfinals_before,semifinals_before,finals_before,winner,finalist,semi_finalist,quarter_finalist
0,2026,France,Europe,0,85,32,25,6,7,2,...,16,10,8,9,7,4,NaN,NaN,NaN,NaN
1,2026,Spain,Europe,0,104,32,29,2,8,1,...,16,11,9,6,2,1,NaN,NaN,NaN,NaN
2,2026,Argentina,South America,0,80,14,30,4,3,3,...,18,15,10,10,6,6,NaN,NaN,NaN,NaN
3,2026,England,Europe,0,82,23,26,6,7,1,...,16,13,8,10,3,1,NaN,NaN,NaN,NaN
4,2026,Portugal,Europe,0,98,31,26,5,7,0,...,8,5,4,3,2,0,NaN,NaN,NaN,NaN
5,2026,Brazil,South America,0,58,39,15,10,10,5,...,22,20,12,17,11,6,NaN,NaN,NaN,NaN
6,2026,Netherlands,Europe,0,92,41,21,8,9,0,...,11,11,9,7,5,3,NaN,NaN,NaN,NaN
7,2026,Morocco,Africa,0,100,18,37,2,9,0,...,6,2,2,1,1,0,NaN,NaN,NaN,NaN
8,2026,Belgium,Europe,0,80,32,19,7,10,0,...,14,9,8,3,2,0,NaN,NaN,NaN,NaN
9,2026,Germany,Europe,0,80,47,21,10,7,4,...,20,18,10,16,13,8,NaN,NaN,NaN,NaN
